In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import MinMaxScaler

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, SimpleRNN

from tensorflow.keras import Input

In [ ]:
df = pd.read_csv("Google_Stock_Price.csv", thousands = ",")


In [ ]:
data = df['Open'].values.reshape(-1,1)
data = pd.to_numeric(df['Open'], errors = 'coerce').dropna().values.reshape(-1,1)

In [ ]:
scaler = MinMaxScaler(feature_range=(0,1))
data_scaled = scaler.fit_transform(data)


In [ ]:
train_size = int(len(data_scaled)*0.8)
train_data = data_scaled[:train_size]
test_data = data_scaled[train_size:]

In [ ]:
def create_dataset(dataset):
  x = []
  y = []

  for i in range(60,len(dataset)):
    x.append(dataset[i-60:i,0])
    y.append(dataset[i,0])

  return np.array(x) , np.array(y)

In [ ]:
X_train, y_train = create_dataset(train_data)
X_test, y_test = create_dataset(test_data)

In [ ]:
X_train = np.reshape(X_train,(X_train.shape[0], X_train.shape[1], 1))
X_test = np.reshape(X_test,(X_test.shape[0], X_test.shape[1], 1))

In [ ]:
model = Sequential([
    Input(shape=(60,1)),
    SimpleRNN(50, return_sequences=True),
    SimpleRNN(50),
    Dense(1)
])

In [ ]:
model.compile(
    optimizer = 'adam',
    loss = 'mse'
)

In [ ]:
model.fit(X_train, y_train, epochs=20, batch_size = 32 )

In [ ]:
predicted = model.predict(X_test)
predicted = scaler.inverse_transform(predicted)
real = scaler.inverse_transform(y_test.reshape(-1,1))

In [ ]:
plt.plot(real , color = 'red', label='real price')
plt.plot(predicted, color='blue', label = 'predicted price')
plt.title("google stock price predication")
plt.xlabel('time')
plt.ylabel('price')
plt.show()